In [ ]:
import time, os 
import requests as http_requests

NOTEBOOK_NAME = "anime-omni-main"
STEP_NAME = "step_omni"

# Carregar secrets
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _ks(n):
        try: return _s.get_secret(n)
        except: return ""
    DATABASE_URL = _ks("DATABASE_URL")
    PROJECT_ID = _ks("PIPELINE_PROJECT_ID")
    PIPELINE_WEBHOOK_URL = _ks("PIPELINE_WEBHOOK_URL")
    TASK_KEY = _ks("PIPELINE_TASK_KEY")
except:
    DATABASE_URL = os.getenv("DATABASE_URL", "")
    PROJECT_ID = os.getenv("PIPELINE_PROJECT_ID", "")
    PIPELINE_WEBHOOK_URL = os.getenv("PIPELINE_WEBHOOK_URL", "")
    TASK_KEY = os.getenv("PIPELINE_TASK_KEY", os.getenv("TASK_KEY", ""))

if not TASK_KEY:
    TASK_KEY = "omni"

def should_run(step):
    """Decide se um passo (translate, tts, assemble) deve ser executado."""
    if TASK_KEY in ["omni", "omni-main", ""]:
        return True
    if step == "translate":
        return TASK_KEY in ["omni", "omni-main", ""]
    if step == "tts":
        return TASK_KEY in ["omni", "omni-main", ""] or TASK_KEY.startswith("omni-tts")
    if step == "assemble":
        return TASK_KEY in ["omni", "omni-main", "omni-assemble"]
    return True

_cell_timers = {}

def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2
        conn = psycopg2.connect(DATABASE_URL)
        cur = conn.cursor()
        cur.execute(query, params)
        conn.commit()
        cur.close()
        conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))
    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))

def fetch_project_opts():
    if not DATABASE_URL or not PROJECT_ID: return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    try:
        import psycopg2
        from psycopg2.extras import RealDictCursor
        conn = psycopg2.connect(DATABASE_URL, cursor_factory=RealDictCursor)
        cur = conn.cursor()
        cur.execute("SELECT bg_audio, srt_type, azure_enabled FROM pipeline_projects WHERE id=%s::uuid", (PROJECT_ID,))
        row = cur.fetchone()
        cur.close()
        conn.close()
        return dict(row) if row else {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    except Exception as e:
        print(f"Erro ao buscar opts: {e}")
        return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}

PROJECT_OPTS = fetch_project_opts()
AZURE_ENABLED = PROJECT_OPTS.get("azure_enabled", True)
print("Tracking de celulas e opts configurados!", PROJECT_OPTS)


In [ ]:
cell_start(1, "Celula 1")
# @title 🚀 1. Setup Inicial: Kaggle + Google Drive OAuth + Whisper + Modelos v6
# @markdown Configure TUDO aqui antes de rodar qualquer outra célula.

import os, sys, shutil, time, asyncio, json, nest_asyncio, subprocess, urllib.request, io, gc
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- 1. INSTALAÇÃO DE DEPENDÊNCIAS ---
print("📦 Instalando bibliotecas (pode demorar 2-3 min)...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")
os.system("pip install python-dotenv faster-whisper gradio_client google-genai edge-tts ffmpeg-python pydub openai nest_asyncio omnivoice torchaudio google-auth google-auth-httplib2 google-api-python-client stable-ts funasr modelscope demucs silero-vad onnxruntime pyannote.audio speechbrain scipy > /dev/null 2>&1")

import torch
import stable_whisper
from google import genai as google_genai_client
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from openai import OpenAI
from pydub import AudioSegment
from dotenv import load_dotenv
nest_asyncio.apply()

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  PAINEL CENTRAL DE CONFIGURAÇÃO — Edite aqui antes de rodar
# ─────────────────────────────────────────────────────────────────────────────



# 🔊 AJUSTE DE VELOCIDADE DE ÁUDIO (Fábrica Cel 6)
TARGET_MIN_SPEED = 1.1
TARGET_MAX_SPEED = 1.35
MAX_AUDIO_RETRIES = 4
MAX_TEXT_RETRIES  = 3

# 🎙️ NARRADOR PADRÃO (usado para busca do áudio de clonagem no Drive)
# O sistema busca arquivos na pasta CLONAGEM/ do Drive cujo nome se pareça com esse.
# Ex: NOME_NARRADOR="Goku" vai achar "goku_dragon_ball.mp3" automaticamente.
NOME_NARRADOR = "Alessandro"  # @param {type:"string"}

# 📦 TRADUÇÃO — TAMANHO DO BATCH (blocos por chamada ao modelo)
# Se o anime foi identificado: usa BATCH_SIZE. Senão: usa 60 (mais conservador).
BATCH_SIZE = 50  # @param {type:"integer"}

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CADEIAS DE MODELOS GEMINI (fallback automático por quota/erro)
#    Adicione modelos extras separados por vírgula — serão tentados em ordem.
# ─────────────────────────────────────────────────────────────────────────────

# Identificação de anime — requer modelo com boa compreensão multilíngue
MODELS_IDENTIFICACAO = [
    "gemini-3.5-flash",
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-2.5-pro"
]

# Tradução em batch — velocidade é importante, flash é suficiente
MODELS_TRADUCAO = [
    "gemini-3.5-flash",
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite",
    "gemini-2.5-pro",
    "gemini-3-flash-preview"
]


# Reescrita de texto da fábrica de áudio — contagem de caracteres
MODELS_REESCRITA = [
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview"
]

# Fallback para OpenAI caso TODOS os Gemini falhem
OPENAI_FALLBACK_MODEL = "gpt-5.4"
USE_OPENAI_FALLBACK   = True  # @param {type:"boolean"}

# ─────────────────────────────────────────────────────────────────────────────
# 🔑 CHAVES DE API — Kaggle Secrets (com fallback para .env local)
# ─────────────────────────────────────────────────────────────────────────────
def _load_credentials():
    # Tenta Kaggle Secrets primeiro (ambiente de produção)
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando chaves do Kaggle Secrets...")
        return _get
    except ImportError:
        pass

    # Fallback: .env local (ambiente de desenvolvimento)
    load_dotenv()
    print("Carregando chaves do .env (desenvolvimento local)...")
    return lambda name: os.getenv(name, "")

_get_secret = _load_credentials()

GEMINI_API_KEY      = _get_secret("GEMINI_API_KEY")
OPENAI_API_KEY      = _get_secret("OPENAI_API_KEY")
DRIVE_ACCESS_TOKEN  = _get_secret("DRIVE_ACCESS_TOKEN")
DRIVE_REFRESH_TOKEN = _get_secret("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID     = _get_secret("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET = _get_secret("DRIVE_CLIENT_SECRET")
HF_TOKEN            = _get_secret("HF_TOKEN")
AZURE_OPENAI_ENDPOINT   = _get_secret("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = _get_secret("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = _get_secret("AZURE_OPENAI_DEPLOYMENT")

_keys_ok = True
for _name, _val in [("GEMINI_API_KEY", GEMINI_API_KEY), ("DRIVE_REFRESH_TOKEN", DRIVE_REFRESH_TOKEN)]:
    if not _val:
        print(f"  ATENCAO: '{_name}' nao encontrada! Verifique Kaggle Secrets.")
        _keys_ok = False
if _keys_ok:
    print("Todas as chaves carregadas com sucesso.")

# ─────────────────────────────────────────────────────────────────────────────
# 📁 CAMINHOS DO PROJETO
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH            = "/kaggle/working/AUDIO_DUB"
TEMP_PATH            = "/kaggle/working/temp_audio"
ANIME_AUDIO_ORIGINAL = "/kaggle/working/input/anime_audio.mp3"
MIN_GAP_CENA_S       = 0.3

MODELOS_OFFLINE       = ""
MODEL_WHISPER_PATH    = "large-v3"
# SenseVoice: prioridade de carregamento
# 1. Dataset Kaggle local (instantâneo)
# 2. HuggingFace (CDN global, rápido)
# 3. ModelScope (servidores China, lento - último recurso)
# Tenta encontrar o SenseVoiceSmall no /kaggle/input de forma case-insensitive
import os as _os
SENSEVOICE_MODEL_PATH = None
if _os.path.exists("/kaggle/input"):
    for _root, _dirs, _files in _os.walk("/kaggle/input"):
        for _d in _dirs:
            if _d.lower() == "sensevoicesmall":
                SENSEVOICE_MODEL_PATH = _os.path.join(_root, _d)
                print(f"[SenseVoice] Encontrado no dataset Kaggle: {SENSEVOICE_MODEL_PATH}")
                break
        if SENSEVOICE_MODEL_PATH:
            break

if not SENSEVOICE_MODEL_PATH:
    # Se não achou localmente, tenta baixar do HuggingFace (muito mais rápido que ModelScope)
    try:
        from huggingface_hub import snapshot_download
        _sv_hf = snapshot_download("FunAudioLLM/SenseVoiceSmall",
                                   local_dir="/kaggle/working/SenseVoiceSmall",
                                   token=HF_TOKEN if HF_TOKEN else None)
        SENSEVOICE_MODEL_PATH = _sv_hf
        print(f"[SenseVoice] Baixado do HuggingFace: {_sv_hf}")
    except Exception as _hf_err:
        print(f"[SenseVoice] HuggingFace falhou ({_hf_err}), usando ModelScope...")
        SENSEVOICE_MODEL_PATH = "iic/SenseVoiceSmall"
        print(f"[SenseVoice] Fallback ModelScope: {SENSEVOICE_MODEL_PATH}")
MODEL_OMNIVOICE_PATH  = "k2-fsa/OmniVoice"

os.chdir("/kaggle/working")
for p in [BASE_PATH, TEMP_PATH, "/kaggle/working/input"]:
    os.makedirs(p, exist_ok=True)

print("Limpando arquivos temporarios...")
for f in os.listdir("/kaggle/working"):
    if f.endswith((".mp3", ".wav", ".srt")):
        try: os.remove(f"/kaggle/working/{f}")
        except: pass

# ─────────────────────────────────────────────────────────────────────────────
# ☁️  GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
print("Autenticando Google Drive...")
try:
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN if DRIVE_ACCESS_TOKEN else None,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID,
        client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("Google Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"Falha na autenticacao do Drive: {e}")

def _buscar_id(caminho_no_drive):
    partes = caminho_no_drive.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        query = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, mimeType)").execute()
        arquivos = results.get("files", [])
        if not arquivos: return None
        parent_id = arquivos[0]["id"]
    return parent_id

def _garantir_pasta(caminho_pasta):
    partes = caminho_pasta.strip("/").split("/")
    parent_id = "root"
    for pasta in partes:
        query = f"name='{pasta}' and '{parent_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        if existentes:
            parent_id = existentes[0]["id"]
        else:
            nova = drive_service.files().create(
                body={"name": pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
                fields="id"
            ).execute()
            parent_id = nova["id"]
    return parent_id

def baixar_do_drive(caminho_no_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        file_id = _buscar_id(caminho_no_drive)
        if not file_id: return False
        request = drive_service.files().get_media(fileId=file_id)
        with open(destino_local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()
        return True
    except: return False

def salvar_no_drive(caminho_local, caminho_destino_drive):
    if drive_service is None or not os.path.exists(caminho_local): return
    try:
        partes = caminho_destino_drive.strip("/").split("/")
        nome_arquivo = partes[-1]
        pasta_drive  = "/".join(partes[:-1]) if len(partes) > 1 else ""
        parent_id = _garantir_pasta(pasta_drive) if pasta_drive else "root"
        query = f"name='{nome_arquivo}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if existentes:
            drive_service.files().update(fileId=existentes[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(
                body={"name": nome_arquivo, "parents": [parent_id]},
                media_body=media, fields="id"
            ).execute()
        print(f"  Salvo no Drive: {caminho_destino_drive}")
    except Exception as e:
        print(f"  Erro ao salvar {caminho_destino_drive}: {e}")

# Download de cache do Drive (Preserva se for um projeto retomado)
has_cached_translation = False
if drive_service:
    print("\nBaixando arquivos de cache do Drive...")
    baixar_do_drive("KAGGLE/AUDIO_DUB/INPUT/anime_audio.mp3",      "/kaggle/working/input/anime_audio.mp3")
    has_trans = baixar_do_drive("KAGGLE/AUDIO_DUB/transcricao_raw.json",        f"{BASE_PATH}/transcricao_raw.json")
    has_trad  = baixar_do_drive("KAGGLE/AUDIO_DUB/traducao_simplificada.json",  f"{BASE_PATH}/traducao_simplificada.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/identificacao_anime.json",    f"{BASE_PATH}/identificacao_anime.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/roteiro_adaptado.json",       f"{BASE_PATH}/roteiro_adaptado.json")
    has_cached_translation = has_trans or has_trad

# Se NÃO havia nenhum cache no Drive (novo projeto ou limpo do zero), remove resquícios locais de containers anteriores
if not has_cached_translation:
    import glob
    print("🧹 Nenhum cache no Drive (Novo Projeto). Limpando resquícios locais de containers anteriores...")
    residual_files = (
        glob.glob("/kaggle/working/roteiro_adaptado*.json") +
        glob.glob("/kaggle/working/*.zip") +
        glob.glob(f"{BASE_PATH}/roteiro_adaptado*.json") +
        glob.glob(f"{BASE_PATH}/*.zip") +
        glob.glob(f"{BASE_PATH}/OUTPUT/*.zip") +
        glob.glob(f"{BASE_PATH}/OUTPUT/roteiro_adaptado*.json")
    )
    for rf in residual_files:
        try:
            os.remove(rf)
            print(f"  🗑️ Removido resquício antigo: {os.path.basename(rf)}")
        except Exception as ex_rf:
            pass
else:
    print("✨ Projeto retomado! Cache de tradução/roteiro carregado com sucesso do Google Drive.")

# --- Busca e download automático do áudio de referência para clonagem ---
REF_AUDIO_PATH = ""
REF_TEXT       = ""

if drive_service and NOME_NARRADOR:
    from difflib import SequenceMatcher
    def _sim(a, b): return SequenceMatcher(None, a.lower(), b.lower()).ratio()

    _clonagem_drive = "KAGGLE/AUDIO_DUB/INPUT/CLONAGEM"
    _clonagem_local = f"{BASE_PATH}/INPUT/CLONAGEM"
    os.makedirs(_clonagem_local, exist_ok=True)

    print(f"Buscando audio de clonagem para '{NOME_NARRADOR}'...")
    _pid = _buscar_id(_clonagem_drive)
    if _pid:
        _res = drive_service.files().list(
            q=f"'{_pid}' in parents and trashed=false and mimeType contains 'audio/'",
            fields="files(id, name)"
        ).execute()
        _melhor, _score = None, 0.0
        for _arq in _res.get('files', []):
            _s = _sim(NOME_NARRADOR, os.path.splitext(_arq['name'])[0])
            if NOME_NARRADOR.lower() in _arq['name'].lower(): _s = max(_s, 0.85)
            if _s > _score: _score, _melhor = _s, _arq
        if _melhor and _score > 0.6:
            print(f"  Match: '{_melhor['name']}' ({_score*100:.1f}%)")
            _local = f"{_clonagem_local}/{_melhor['name']}"
            if not os.path.exists(_local):
                _req = drive_service.files().get_media(fileId=_melhor['id'])
                with open(_local, "wb") as _f:
                    _dl = MediaIoBaseDownload(_f, _req)
                    _done = False
                    while not _done: _, _done = _dl.next_chunk()
            REF_AUDIO_PATH = _local
            print(f"  Referencia de clonagem definida: {REF_AUDIO_PATH}")
        else:
            print(f"  Nenhum match >60% para '{NOME_NARRADOR}' na pasta CLONAGEM.")
    else:
        print("  Pasta CLONAGEM nao encontrada no Drive.")

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CLIENTES DE API
# ─────────────────────────────────────────────────────────────────────────────
client_gemini = google_genai_client.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY.startswith("AIza") else None
client_openai = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY.startswith("sk-") else None
client_azure = None
if AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT:
    print("Inicializando cliente Azure OpenAI...")
    client_azure = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

# ─────────────────────────────────────────────────────────────────────────────
# 🔁 FUNÇÃO CENTRAL DE CHAMADA GEMINI COM FALLBACK POR QUOTA
# ─────────────────────────────────────────────────────────────────────────────
# Erros que indicam quota esgotada ou sobrecarga — deve tentar próximo modelo
QUOTA_ERROR_CODES = {429, 503, 500}
QUOTA_ERROR_MSGS  = [
    "quota", "rate limit", "resource exhausted", "overloaded",
    "too many requests", "service unavailable", "capacity",
    "not found", "not supported", "is not found", "timeout", "408", "deadline",
    "503", "500", "internal", "temporarily", "unavailable", "backoff",
]

def _is_quota_error(e: Exception) -> bool:
    msg = str(e).lower()
    if "psycopg" in msg or "connection" in msg or "port" in msg:
        return False
    code = getattr(e, "code", getattr(e, "status_code", None))
    if code in QUOTA_ERROR_CODES: return True
    return any(m in msg for m in QUOTA_ERROR_MSGS)

def azure_generate(prompt, response_format=None, system_instruction=None):
    if not client_azure:
        raise RuntimeError("client_azure nao inicializado")
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": str(system_instruction)})
    messages.append({"role": "user", "content": str(prompt)})
    try:
        kwargs = {
            "model": AZURE_OPENAI_DEPLOYMENT,
            "messages": messages,
            "temperature": 1.0
        }
        if response_format:
            kwargs["response_format"] = response_format
        resp = client_azure.chat.completions.create(**kwargs)
        return resp.choices[0].message.content
    except Exception as e:
        print(f"  [AZURE] Falha com chat.completions ({e}). Tentando responses.create...")
        resp = client_azure.responses.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            input=prompt
        )
        return resp.output[0]

def gemini_generate(model_chain: list, contents, config: dict = None, task_name: str = ""):
    """
    Tenta gerar conteúdo pelo Gemini (ou Azure OpenAI fallback se habilitado e primeiro falhar).
    """
    global AZURE_ENABLED
    if client_gemini is None:
        raise RuntimeError("client_gemini nao inicializado — verifique GEMINI_API_KEY no .env")

    # 1. Tentar primeiro modelo (geralmente gemini-3.5-flash)
    first_model = model_chain[0]
    try:
        kwargs = {"model": first_model, "contents": contents}
        if config:
            kwargs["config"] = config
        response = client_gemini.models.generate_content(**kwargs)
        print(f"  [OK] {task_name or 'gemini_generate'} -> {first_model}")
        return response
    except Exception as e:
        print(f"  [FAIL] {first_model} falhou: {str(e)[:100]}")
        
        # 2. Se habilitado no bot, tenta Azure OpenAI como segundo na prioridade
        if AZURE_ENABLED and client_azure:
            print(f"  [AZURE] Tentando Azure OpenAI ({AZURE_OPENAI_DEPLOYMENT}) como fallback...")
            try:
                sys_inst = None
                resp_fmt = None
                if config:
                    if isinstance(config, dict):
                        sys_inst = config.get("system_instruction")
                        if config.get("response_mime_type") == "application/json":
                            resp_fmt = {"type": "json_object"}
                    else:
                        sys_inst = getattr(config, "system_instruction", None)
                        if getattr(config, "response_mime_type", None) == "application/json":
                            resp_fmt = {"type": "json_object"}
                
                azure_text = azure_generate(contents, resp_fmt, sys_inst)
                print(f"  [OK] {task_name or 'gemini_generate'} -> Azure ({AZURE_OPENAI_DEPLOYMENT})")
                
                class FakeGeminiResponse:
                    def __init__(self, text):
                        self.text = text
                return FakeGeminiResponse(azure_text)
            except Exception as ae:
                print(f"  [AZURE FAIL] Azure OpenAI falhou: {ae}")
        
        # 3. Prossegue com a cadeia padrão do Gemini
        last_exc = e
        for model in model_chain[1:]:
            try:
                kwargs = {"model": model, "contents": contents}
                if config:
                    kwargs["config"] = config
                response = client_gemini.models.generate_content(**kwargs)
                print(f"  [OK] {task_name or 'gemini_generate'} -> {model}")
                return response
            except Exception as e2:
                if _is_quota_error(e2):
                    print(f"  [QUOTA] {model}: {str(e2)[:80]} — tentando proximo...")
                    last_exc = e2
                    time.sleep(2)
                    continue
                else:
                    raise

        # 4. Fallback final para OpenAI se habilitado
        if USE_OPENAI_FALLBACK and client_openai:
            print(f"  [FALLBACK] Todos os modelos Gemini/Azure falharam. Usando OpenAI {OPENAI_FALLBACK_MODEL}...")
            raise RuntimeError(
                f"OPENAI_FALLBACK: Todos Gemini/Azure esgotados. "
                f"Ultimo erro: {last_exc}. "
                f"Use client_openai com modelo {OPENAI_FALLBACK_MODEL} no bloco catch da celula chamadora."
            )

        raise RuntimeError(f"Todos os modelos Gemini/Azure falharam. Ultimo erro: {last_exc}")

# ─────────────────────────────────────────────────────────────────────────────
# 🎤 WHISPER (stable-ts + faster-whisper)
# ─────────────────────────────────────────────────────────────────────────────
GPU_COUNT    = torch.cuda.device_count()
device       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"

print(f"\nGPUs disponíveis: {GPU_COUNT}")
print(f"Carregando Whisper em {device}:0 ({COMPUTE_TYPE})...")
if False:
    print("ERRO: Caminho do Whisper nao encontrado!")
else:
    stable_model = stable_whisper.load_faster_whisper(
        MODEL_WHISPER_PATH,
        device=device,
        device_index=0,        # GPU0 explícito (ctranslate2 não aceita "cuda:0")
        compute_type=COMPUTE_TYPE,
    )
    print("Whisper carregado em GPU0!")

# Preenchido pela Cel8 após subir servidores OmniVoice
OMNIVOICE_PORTS = []

# ─────────────────────────────────────────────────────────────────────────────
# 📋 RESUMO DA CONFIGURACAO
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
=================================================================
  CONFIGURACAO DO PIPELINE
=================================================================
  Velocidade alvo : {TARGET_MIN_SPEED}x - {TARGET_MAX_SPEED}x
  Fallback OpenAI : {USE_OPENAI_FALLBACK} ({OPENAI_FALLBACK_MODEL})
-----------------------------------------------------------------
  Identificacao   : {' > '.join(MODELS_IDENTIFICACAO)}
  Traducao        : {' > '.join(MODELS_TRADUCAO)}
  Reescrita       : {' > '.join(MODELS_REESCRITA)}
=================================================================
""")

print("SETUP CONCLUIDO! Execute a Fase 5 (Servidores OmniVoice) antes de rodar a Fabrica de Audio.")

cell_end(1, "done", "Celula 1 concluido")

In [ ]:
cell_start(2, "Setup Inicial + Drive + Whisper")
# =============================================================================
# @title 🎙️ 2. Transcrição v6 (Demucs + stable-ts + SenseVoice + Pyannote)
# @markdown Isolamento vocal, transcrição por região, classificação de idioma,
# @markdown segunda passagem para falas perdidas, e anti-alucinação.
# =============================================================================

import json, os, re, subprocess
import numpy as np
from pathlib import Path
from pydub import AudioSegment
from tqdm.notebook import tqdm
from funasr import AutoModel

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️ CONFIGURAÇÕES
# ─────────────────────────────────────────────────────────────────────────────
TRANSCRICAO_PATH = f"{BASE_PATH}/transcricao_raw.json"

WHISPER_BEAM     = 5
MAX_SEG_WORDS    = 25

VAD_MIN_SILENCE_MS = 100
VAD_SPEECH_PAD_MS  = 60
VAD_THRESHOLD      = 0.25

VAD2_MIN_SILENCE_MS = 50
VAD2_SPEECH_PAD_MS  = 40
VAD2_THRESHOLD      = 0.15

VAD_TRIM_ONSET_PAD_MS  = 20
VAD_TRIM_OFFSET_PAD_MS = 30

MERGE_ZH_GAP_S   = 1.00
MERGE_JA_GAP_S   = 0.25
MIN_BLOCK_DUR_S  = 0.12
MAX_CHARS_PER_SEC = 18.0

SCAN_GAP_MIN_S    = 0.4
SCAN_SILERO_SENS  = 0.15

WATERMARK_PATTERNS = [
    "cảm ơn", "theo dõi", "hẹn gặp", "thank you for watching",
    "subscribe", "like and", "don't forget", "ご視聴", "チャンネル登録", "高評価",
    "다음 영상에서 만나요.", "ご視聴ありがとうございました", "また会いましょう", "지금까지 흔더리",
]

if not os.path.exists(ANIME_AUDIO_ORIGINAL):
    raise FileNotFoundError(f"❌ Áudio não encontrado: {ANIME_AUDIO_ORIGINAL}")

# =============================================================================
# 🔤 FUNÇÕES AUXILIARES
# =============================================================================
def analyze_script(text: str) -> dict:
    counts = {"hiragana": 0, "katakana": 0, "cjk": 0, "latin": 0, "other": 0}
    for ch in text:
        cp = ord(ch)
        if   0x3040 <= cp <= 0x309F: counts["hiragana"] += 1
        elif 0x30A0 <= cp <= 0x30FF: counts["katakana"] += 1
        elif (0x4E00 <= cp <= 0x9FFF) or (0x3400 <= cp <= 0x4DBF):
            counts["cjk"] += 1
        elif ch.isascii() and ch.isalpha(): counts["latin"] += 1
        elif not ch.isspace(): counts["other"] += 1
    total = sum(counts.values()) or 1
    kana  = counts["hiragana"] + counts["katakana"]
    if kana > 0:
        return {"lang": "ja", "confidence": min(1.0, 0.78 + (kana/total)*0.22)}
    if counts["cjk"] > 0:
        return {"lang": "zh", "confidence": min(1.0, 0.65 + (counts["cjk"]/total)*0.35)}
    if counts["latin"] > 0:
        return {"lang": "other", "confidence": 0.55}
    return {"lang": "ambiguous", "confidence": 0.25}

def is_hallucination(text: str, start: float, end: float) -> bool:
    dur = end - start
    text_lower = text.strip().lower()
    if any(pat in text_lower for pat in WATERMARK_PATTERNS): return True
    if dur > 0 and len(text) / dur > MAX_CHARS_PER_SEC: return True
    if len(text) <= 2 and dur < 0.4: return True
    return False

# =============================================================================
# 🔬 SILERO VAD
# =============================================================================
print("🔬 Carregando Silero VAD...")
try:
    from silero_vad import load_silero_vad, get_speech_timestamps
    _silero_model = load_silero_vad()
    SILERO_OK = True
    print("   ✅ Silero VAD pronto.")
except Exception:
    SILERO_OK = False
    print("   ⚠️  Silero VAD não disponível.")

def vad_trim(audio_full: AudioSegment, start_s: float, end_s: float) -> tuple:
    if not SILERO_OK: return start_s, end_s
    ms_s = max(0, int(start_s * 1000))
    ms_e = min(len(audio_full), int(end_s * 1000))
    clip = audio_full[ms_s:ms_e]
    if len(clip) < 80: return start_s, end_s
    clip_16k = clip.set_frame_rate(16000).set_channels(1)
    raw = np.array(clip_16k.get_array_of_samples(), dtype=np.float32) / 32768.0
    try:
        ts = get_speech_timestamps(raw, _silero_model, sampling_rate=16000, min_speech_duration_ms=50, min_silence_duration_ms=40)
    except Exception:
        return start_s, end_s
    if not ts: return start_s, end_s
    onset_ms  = max(0,        ts[0]["start"] / 16.0 - VAD_TRIM_ONSET_PAD_MS)
    offset_ms = min(len(clip), ts[-1]["end"]  / 16.0 + VAD_TRIM_OFFSET_PAD_MS)
    new_start = round(start_s + onset_ms  / 1000.0, 4)
    new_end   = round(start_s + offset_ms / 1000.0, 4)
    if new_end - new_start < 0.05: return start_s, end_s
    return new_start, new_end

# =============================================================================
# VERIFICACAO DE CACHE
# =============================================================================
if os.path.exists(TRANSCRICAO_PATH):
    print(f"Transcricao encontrada. Carregando cache...")
    with open(TRANSCRICAO_PATH, 'r', encoding='utf-8') as f:
        final_blocks = json.load(f)
    print(f"{len(final_blocks)} blocos carregados.")

else:
    # =====================================================================
    # 1/4 DEMUCS + PYANNOTE
    # 2 GPUs: Pyannote roda em GPU1 em paralelo com Demucs em GPU0.
    # 1 GPU:  sequencial (Demucs primeiro, depois Pyannote).
    # =====================================================================
    _gpu_count    = globals().get("GPU_COUNT", torch.cuda.device_count())
    _use_parallel = (_gpu_count >= 2)

    DIAR_SCRIPT = "/kaggle/working/_run_diar.py"
    DIAR_OUTPUT = "/kaggle/working/_diar_result.json"
    DIAR_WAV    = "/kaggle/working/_temp_diar.wav"

    # Pyannote/torchaudio tem bugs com MP3/M4A ("requested chunk resulted in x samples instead of y")
    # Resolvemos pre-convertendo para WAV mono 16k (tambem acelera o Pyannote brutalmente)
    subprocess.run(["ffmpeg", "-y", "-i", ANIME_AUDIO_ORIGINAL, "-ar", "16000", "-ac", "1", DIAR_WAV], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Gera o script de diarizacao usando lista de strings (evita conflito de aspas)
    _lines = [
        "import sys, types, torch, torchaudio\n",
        "if not hasattr(torchaudio, 'set_audio_backend'):\n",
        "    torchaudio.set_audio_backend = lambda x: None\n",
        "if not hasattr(torchaudio, 'get_audio_backend'):\n",
        "    torchaudio.get_audio_backend = lambda: 'soundfile'\n",
        "if 'torchaudio.backend' not in sys.modules:\n",
        "    sys.modules['torchaudio.backend'] = types.ModuleType('torchaudio.backend')\n",
        "if 'torchaudio.backend.common' not in sys.modules:\n",
        "    mod = types.ModuleType('torchaudio.backend.common')\n",
        "    mod.AudioMetaData = getattr(torchaudio, 'AudioMetaData', type('AudioMetaData', (), {}))\n",
        "    sys.modules['torchaudio.backend.common'] = mod\n",
        "import json\n",
        "from pyannote.audio import Pipeline\n",
        f"pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token='{HF_TOKEN}')\n",
        "pipeline = pipeline.to(torch.device('cuda'))\n",
        f"diarization = pipeline('{DIAR_WAV}')\n",
        "if hasattr(diarization, 'speaker_diarization'): diarization = diarization.speaker_diarization\n",
        "segments = []\n",
        "for turn, _, speaker in diarization.itertracks(yield_label=True):\n",
        "    segments.append({'start': round(turn.start,4), 'end': round(turn.end,4), 'speaker': speaker})\n",
        f"with open('{DIAR_OUTPUT}', 'w') as out: json.dump(segments, out)\n",
        "print(f'Done {len(segments)} segmentos.', flush=True)\n",
    ]
    with open(DIAR_SCRIPT, "w", encoding="utf-8") as _df:
        _df.writelines(_lines)

    proc_diar = None
    if _use_parallel:
        _gpu1_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "1"}
        print("   Pyannote iniciado em GPU1 (paralelo com Demucs)...")
        proc_diar = subprocess.Popen(
            ["python", DIAR_SCRIPT],
            env=_gpu1_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
    else:
        print("   1 GPU detectada - Pyannote rodara apos Demucs.")

    _demucs_device = "cuda:0" if _use_parallel else "cuda"
    print(f"\n1/4  Demucs em {_demucs_device}...")
    subprocess.run(
        ["python", "-m", "demucs", "--two-stems", "vocals",
         "-n", "htdemucs", "-d", _demucs_device,
         "--out", "/kaggle/working/demucs_out", ANIME_AUDIO_ORIGINAL],
        check=True
    )
    stem_name        = Path(ANIME_AUDIO_ORIGINAL).stem
    VOCAL_PATH       = f"/kaggle/working/demucs_out/htdemucs/{stem_name}/vocals.wav"
    audio_full       = AudioSegment.from_file(VOCAL_PATH)
    total_duration_s = len(audio_full) / 1000.0
    print(f"   Vocais isolados: {total_duration_s:.1f}s")

    # ==========================================================================
    # 2/4 MODELOS AUXILIARES
    # ==========================================================================
    print("\n2/4  Carregando SenseVoice...")
    sense_model = AutoModel(model=SENSEVOICE_MODEL_PATH, device="cuda:0", disable_update=True)
    print("   SenseVoice Small carregado.")

    # Coleta ou roda Pyannote
    print("   Rodando/aguardando Pyannote...")
    if proc_diar is not None:
        _dout, _derr = proc_diar.communicate()   # aguarda e captura de uma vez
        _diar_ok = (proc_diar.returncode == 0)
        if not _diar_ok:
            print(f"   Pyannote (GPU1) falhou:\n{_derr.decode()[-1200:]}")
    else:
        _r = subprocess.run(["python", DIAR_SCRIPT], capture_output=True, text=True)
        _diar_ok = (_r.returncode == 0)
        if not _diar_ok:
            print(f"   Pyannote falhou:\n{_r.stderr[-1200:]}")

    if _diar_ok and os.path.exists(DIAR_OUTPUT):
        with open(DIAR_OUTPUT) as f:
            diar_segments = json.load(f)
        print(f"   {len(diar_segments)} segmentos de diarizacao coletados.")
    else:
        print("   Continuando sem agrupamento de speakers.")
        diar_segments = []


    def merge_diar_regions(diar_segs, gap_tol=0.4):
        if not diar_segs: return []
        s = sorted(diar_segs, key=lambda d: d["start"])
        merged = [dict(s[0])]
        for d in s[1:]:
            last = merged[-1]
            if d["speaker"] == last["speaker"] and (d["start"] - last["end"]) <= gap_tol:
                last["end"] = max(last["end"], d["end"])
            else:
                merged.append(dict(d))
        return merged

    speaker_regions = merge_diar_regions(diar_segments)
    full_regions = []
    cursor = 0.0
    for r in sorted(speaker_regions, key=lambda x: x["start"]):
        if r["start"] - cursor > 0.3:
            full_regions.append({"start": cursor, "end": r["start"], "speaker": "unknown"})
        full_regions.append(r)
        cursor = r["end"]
    if total_duration_s - cursor > 0.3:
        full_regions.append({"start": cursor, "end": total_duration_s, "speaker": "unknown"})

    # ==========================================================================
    # 🧠 TRANSCRIÇÃO POR REGIÃO
    # ==========================================================================
    TEMP_WAV = "/kaggle/working/_temp_region.wav"

    def transcribe_region(start_s, end_s, language=None, vad_min_silence_ms=VAD_MIN_SILENCE_MS, vad_speech_pad_ms=VAD_SPEECH_PAD_MS, threshold=VAD_THRESHOLD):
        ms_s = max(0, int(start_s * 1000))
        ms_e = min(len(audio_full), int(end_s * 1000))
        clip = audio_full[ms_s:ms_e]
        if len(clip) < 200: return []
        clip.export(TEMP_WAV, format="wav")
        try:
            if 'stable_model' not in globals(): raise RuntimeError('Whisper nao carregou. Verifique MODEL_WHISPER_PATH.')
            result = stable_model.transcribe_stable(
                TEMP_WAV, language=language, beam_size=WHISPER_BEAM, word_timestamps=True,
                vad_filter=True, vad_parameters=dict(min_silence_duration_ms=vad_min_silence_ms, speech_pad_ms=vad_speech_pad_ms, threshold=threshold),
                temperature=0, condition_on_previous_text=False, no_speech_threshold=0.3,
                log_prob_threshold=-1.5, compression_ratio_threshold=2.4,
                regroup=True, suppress_silence=True, only_voice_freq=True,
            )
        except Exception as e:
            print(f"      ⚠️ Erro [{start_s:.1f}-{end_s:.1f}s]: {e}")
            return []
        segs = []
        for seg in result.segments:
            text = (seg.text or "").strip()
            if not text: continue
            words = []
            if hasattr(seg, "words") and seg.words:
                for w in seg.words:
                    words.append({"word": w.word, "start": round(start_s + w.start, 4), "end": round(start_s + w.end, 4)})
            seg_start = words[0]["start"] if words else round(start_s + seg.start, 4)
            seg_end   = words[-1]["end"]  if words else round(start_s + seg.end,   4)
            seg_start = max(start_s, seg_start)
            seg_end   = min(end_s,   seg_end)
            if seg_end <= seg_start: continue
            segs.append({"text": text, "start": seg_start, "end": seg_end, "words": words, "avg_logprob": getattr(seg, "avg_logprob", 0.0), "detected_lang": result.language if hasattr(result, "language") else None})
        return segs

    # ==========================================================================
    # 🎤 3/4 TRANSCRIÇÃO
    # ==========================================================================
    print(f"\n🎤 3/4  Transcrevendo por região...")
    all_raw_segs = []
    for region in tqdm(full_regions, desc="Transcrevendo"):
        if region["end"] - region["start"] < 0.3: continue
        segs = transcribe_region(region["start"], region["end"])
        for s in segs:
            s["speaker"] = region["speaker"]
            all_raw_segs.append(s)
    print(f"   → {len(all_raw_segs)} segmentos brutos.")

    PHRASE_PUNCT = set("。！？…」』）")
    def split_long_segment(seg_words, max_words):
        if len(seg_words) <= max_words: return [seg_words]
        groups, current = [], []
        for w in seg_words:
            current.append(w)
            if any(ch in PHRASE_PUNCT for ch in w["word"].strip()) or len(current) >= max_words:
                groups.append(current); current = []
        if current:
            if groups and len(current) <= 3: groups[-1].extend(current)
            else: groups.append(current)
        return groups or [seg_words]

    # ==========================================================================
    # 🔀 4/4 CONSOLIDAÇÃO
    # ==========================================================================
    print("\n🧩 4/4  Consolidando...")
    TEMP_CHUNK = "/kaggle/working/_temp_chunk.wav"

    # ── FIX: coleta todos os blocos antes de ordenar ─────────────────────────
    # ORIGINAL: resolved.sort() era chamado dentro do loop a cada iteração.
    # Isso é inofensivo mas mascarava o problema real: blocos gerados por regiões
    # diferentes podiam ter timestamps que se sobrepõem (artefato do Whisper em
    # narrações chinesas com frases repetidas). A ordenação prematura reordenava
    # sem resolver as sobreposições, que chegavam intactas ao final_blocks.
    # CORREÇÃO: coleta completa → sort único → resolve_overlaps → merge_adjacent.
    # ─────────────────────────────────────────────────────────────────────────
    resolved_raw = []
    for seg in tqdm(all_raw_segs, desc="Processando"):
        text = seg["text"].strip()
        if not text: continue
        words = seg.get("words", [])
        sub_groups = split_long_segment(words, MAX_SEG_WORDS) if words else [None]
        for grp in sub_groups:
            if grp:
                sub_text = "".join(w["word"] for w in grp).strip()
                sub_start, sub_end = grp[0]["start"], grp[-1]["end"]
            else:
                sub_text, sub_start, sub_end = text, seg["start"], seg["end"]
            if not sub_text or is_hallucination(sub_text, sub_start, sub_end): continue
            sub_start, sub_end = vad_trim(audio_full, sub_start, sub_end)
            if sub_end - sub_start < MIN_BLOCK_DUR_S: continue
            r = analyze_script(sub_text)
            if r["lang"] == "ja":
                lang_final, conf = "ja", r["confidence"]
            elif r["lang"] == "zh":
                # SenseVoice: classifica se o segmento zh é realmente ja ou zh/yue.
                # Não gera transcrição nova — apenas retorna a tag de idioma detectado.
                audio_full[max(0,int(sub_start*1000)):min(len(audio_full),int(sub_end*1000))].export(TEMP_CHUNK, format="wav")
                try:
                    sv_res = sense_model.generate(input=TEMP_CHUNK, language="auto", use_itn=False)
                    m = re.search(r'<\|(zh|ja|yue|ko|en|nospeech)\|>', sv_res[0]["text"])
                    sv_tag = m.group(1) if m else "unknown"
                except Exception:
                    sv_tag = "unknown"
                if sv_tag == "ja" and (r["confidence"] < 0.8 or len(sub_text) < 6):
                    lang_final, conf = "ja", 0.65
                elif sv_tag in ("zh", "yue"):
                    lang_final, conf = "zh", min(1.0, r["confidence"] + 0.12)
                else:
                    lang_final, conf = "zh", r["confidence"] * 0.9
            else:
                lang_final, conf = "other", r["confidence"]
            tipo = "NARRACAO" if lang_final == "zh" else "CENA"
            resolved_raw.append({"tipo": tipo, "start": round(sub_start, 4), "end": round(sub_end, 4), "text": sub_text, "lang_final": lang_final, "confidence": round(conf, 3), "logprob": round(seg.get("avg_logprob", 0.0), 3), "speaker": seg.get("speaker", "?")})

    # Ordena uma única vez após coleta completa
    resolved_raw.sort(key=lambda b: b["start"])

    # ==========================================================================
    # FIX: resolve_overlaps — remove sobreposições temporais entre blocos
    #
    # PROBLEMA: O Whisper em narrações chinesas com frases repetidas ("如同神明降临
    # 如同神明降临") gera segmentos cujo start é anterior ao end do segmento anterior.
    # Ex: bloco A: [10.6s-17.0s], bloco B: [14.9s-20.2s] → B começa DENTRO de A.
    #
    # ESTRATÉGIA:
    #   - Se B.start < A.end → há sobreposição real
    #   - Ajusta B.start para A.end (clipa o início do bloco invadido)
    #   - Se após o ajuste B ficar menor que MIN_BLOCK_DUR_S → descarta B
    #   - Se B estiver 100% contido em A (B.end <= A.end) → descarta B
    #
    # Mantém a função pura (recebe lista, retorna lista nova) para reuso.
    # ==========================================================================
    def resolve_overlaps(blocks):
        """
        Resolve sobreposições temporais entre blocos consecutivos.
        Whisper pode gerar segmentos com start anterior ao end do segmento
        anterior (artefato em narrações zh com frases repetidas).

        Estratégia: ajusta start do bloco seguinte para o end do anterior.
        Se o resultado for menor que MIN_BLOCK_DUR_S ou bloco completamente
        coberto, descarta o bloco sobrepositor.
        Retorna nova lista — não modifica a entrada.
        """
        if not blocks:
            return blocks
        result = [dict(blocks[0])]
        sobr_count = 0
        for curr in blocks[1:]:
            prev = result[-1]
            if curr["start"] < prev["end"]:
                sobr_count += 1
                new_start = prev["end"]
                # Bloco completamente coberto → descarta
                if new_start >= curr["end"]:
                    continue
                # Resultado muito curto após ajuste → descarta
                if (curr["end"] - new_start) < MIN_BLOCK_DUR_S:
                    continue
                # Ajusta start e mantém o bloco
                curr = dict(curr)
                curr["start"] = round(new_start, 4)
                # Filtra as words que ainda estão dentro da janela ajustada
                if curr.get("words"):
                    curr["words"] = [w for w in curr["words"] if w["end"] > new_start]
            result.append(curr)
        if sobr_count:
            print(f"   ⚠️  resolve_overlaps: {sobr_count} sobreposição(ões) corrigida(s).")
        return result

    def merge_adjacent(blocks):
        if not blocks:
            return blocks
        # Conjunto de pontuação final de frase
        END_PUNCT = set(". ! ? 。！？…")
        out = [blocks[0]]
        for curr in blocks[1:]:
            prev = out[-1]
            gap = curr["start"] - prev["end"]
            prev_ends_punct = prev["text"].strip()[-1] in END_PUNCT if prev["text"].strip() else False

            if prev["tipo"] == "NARRACAO" == curr["tipo"] and gap <= MERGE_ZH_GAP_S and (prev["end"] - prev["start"]) < 5.0 and not prev_ends_punct:
                prev["text"] = prev["text"] if prev["text"] == curr["text"] else f"{prev['text']}{curr['text']}"
                prev["end"] = curr["end"]
                prev["confidence"] = (prev["confidence"] + curr["confidence"]) / 2
                continue

            if prev["tipo"] == "CENA" == curr["tipo"] and gap <= MERGE_JA_GAP_S and prev.get("speaker") == curr.get("speaker") and (prev["end"] - prev["start"]) < 2.0 and not prev_ends_punct:
                prev["text"] = f"{prev['text']}{curr['text']}"
                prev["end"] = curr["end"]
                continue

            out.append(curr)
        return out

    # Pipeline: resolve sobreposições → funde adjacentes → exibe contagem
    resolved = resolve_overlaps(resolved_raw)
    resolved = merge_adjacent(resolved)
    print(f"   → {len(resolved)} blocos após resolve+fusão.")

    # ==========================================================================
    # 🔍 3.5 SEGUNDA PASSAGEM — gaps sem fala detectada
    # ==========================================================================
    print("\n🔍 3.5  Segunda passagem nos gaps...")

    # Calcula gaps usando cursor que respeita o end de cada bloco (sem retroagir)
    all_times = sorted(resolved, key=lambda x: x["start"])
    gaps = []
    cursor = 0.0
    for blk in all_times:
        if blk["start"] - cursor >= SCAN_GAP_MIN_S: gaps.append((cursor, blk["start"]))
        cursor = max(cursor, blk["end"])
    if total_duration_s - cursor >= SCAN_GAP_MIN_S: gaps.append((cursor, total_duration_s))
    print(f"   → {len(gaps)} gaps.")

    new_segs = []
    for (gs, ge) in gaps:
        clip = audio_full[max(0,int(gs*1000)):min(len(audio_full),int(ge*1000))]
        if len(clip) < 200: continue
        clip_16k = clip.set_frame_rate(16000).set_channels(1)
        raw = np.array(clip_16k.get_array_of_samples(), dtype=np.float32) / 32768.0
        try:
            speech_ts = get_speech_timestamps(raw, _silero_model, sampling_rate=16000, min_speech_duration_ms=80, min_silence_duration_ms=60, threshold=SCAN_SILERO_SENS)
        except Exception:
            continue
        if not speech_ts: continue
        for sp in speech_ts:
            onset_s = gs + sp["start"] / 16000.0
            offset_s = gs + sp["end"] / 16000.0
            if offset_s - onset_s < 0.2: continue
            segs = transcribe_region(onset_s, offset_s, language="ja", vad_min_silence_ms=VAD2_MIN_SILENCE_MS, vad_speech_pad_ms=VAD2_SPEECH_PAD_MS, threshold=VAD2_THRESHOLD)
            for s in segs:
                if is_hallucination(s["text"], s["start"], s["end"]): continue
                r = analyze_script(s["text"])
                if r["lang"] == "ja" or (r["lang"] == "zh" and r["confidence"] < 0.8):
                    trim_s, trim_e = vad_trim(audio_full, s["start"], s["end"])
                    if trim_e - trim_s < MIN_BLOCK_DUR_S: continue
                    new_segs.append({"tipo": "CENA", "start": round(trim_s, 4), "end": round(trim_e, 4), "text": s["text"], "lang_final": "ja", "confidence": round(max(0.6, r["confidence"]), 3), "logprob": round(s.get("avg_logprob", 0.0), 3), "speaker": "unknown"})

    print(f"   → {len(new_segs)} novos segmentos.")
    resolved.extend(new_segs)
    resolved.sort(key=lambda b: b["start"])

    # Aplica resolve_overlaps também após a segunda passagem (novos segs podem sobrepor existentes)
    resolved = resolve_overlaps(resolved)
    resolved = merge_adjacent(resolved)

    # Limpeza de watermarks
    HALL_PHRASES = {"ご視聴ありがとうございました", "ご視聴ありがとうございました。", "ご視聴ありがとうございます", "また会いましょう", "다음 영상에서 만나요.", "지금까지 흔더리", "Thanks for watching!", "Thank you for watching!", "チャンネル登録お願いします", "高評価お願いします", "please like and subscribe", "subscribe to my channel", "don't forget to subscribe", "to show"}
    before = len(resolved)
    resolved = [b for b in resolved if not (b["end"]-b["start"] < 2.5 and b.get("confidence",0) < 0.95 and any(p in b.get("text","") for p in HALL_PHRASES))]
    print(f"   🗑️  {before - len(resolved)} watermarks removidos.")

    # ==========================================================================
    # 🎬 TIMELINE FINAL
    # ==========================================================================
    final_blocks = []
    block_id = 0
    cursor = 0.0
    for seg in resolved:
        gap = seg["start"] - cursor

        # ── FIX: guard contra sobreposição residual ───────────────────────────
        # Após resolve_overlaps o ideal é não ter mais sobreposições, mas como
        # segunda linha de defesa verificamos aqui também. Se seg.start < cursor,
        # tentamos ajustar. Se o resultado ficar abaixo de MIN_BLOCK_DUR_S, pula.
        if seg["start"] < cursor:
            new_start = cursor
            if new_start >= seg["end"] or (seg["end"] - new_start) < MIN_BLOCK_DUR_S:
                continue   # bloco completamente coberto ou muito curto → descarta
            seg = dict(seg)
            seg["start"] = round(new_start, 4)
            gap = 0.0     # recalcula gap após ajuste
        # ─────────────────────────────────────────────────────────────────────

        if gap >= MIN_GAP_CENA_S:
            final_blocks.append({"id": block_id, "tipo": "CENA", "start": round(cursor, 4), "end": round(seg["start"], 4), "texto_original": "", "translated_text": "", "status": "CENA_SEM_FALA"})
            block_id += 1
        final_blocks.append({"id": block_id, "tipo": seg["tipo"], "start": round(seg["start"], 4), "end": round(seg["end"], 4), "texto_original": seg["text"], "translated_text": "", "status": "PENDING", "detected_lang": seg["lang_final"], "confidence": seg["confidence"], "avg_logprob": seg["logprob"], "speaker": seg.get("speaker", "?")})
        block_id += 1
        cursor = seg["end"]
    if total_duration_s - cursor >= MIN_GAP_CENA_S:
        final_blocks.append({"id": block_id, "tipo": "CENA", "start": round(cursor, 4), "end": round(total_duration_s, 4), "texto_original": "", "translated_text": "", "status": "CENA_SEM_FALA"})

    # ✅ PÓS-PROCESSAMENTO: CENA_SEM_FALA < 1s → descarta e estende o bloco anterior
    cleaned = []
    for blk in final_blocks:
        if blk.get("status") == "CENA_SEM_FALA" and (blk["end"] - blk["start"]) < 1.0:
            if cleaned:
                cleaned[-1]["end"] = blk["end"]
        else:
            cleaned.append(blk)
    for i, b in enumerate(cleaned):
        b["id"] = i
    final_blocks = cleaned

    with open(TRANSCRICAO_PATH, "w", encoding="utf-8") as f:
        json.dump(final_blocks, f, ensure_ascii=False, indent=4)

    n_nar = sum(1 for b in final_blocks if b["tipo"] == "NARRACAO")
    n_cena = sum(1 for b in final_blocks if b["tipo"] == "CENA" and b.get("status") == "PENDING")
    n_sil = sum(1 for b in final_blocks if b.get("status") == "CENA_SEM_FALA")
    print(f"\n✅ TRANSCRIÇÃO v6 CONCLUÍDA: {n_nar} NARRACAO | {n_cena} CENA c/ fala | {n_sil} CENA sem fala | {len(final_blocks)} total")
    print(f"💾 Salvo em: {TRANSCRICAO_PATH}")

# Preview
print("\n🔎 Preview (Top 10):")
for b in final_blocks[:10]:
    lp = f" [lp={b.get('avg_logprob', '')}]" if b.get("avg_logprob") else ""
    txt = b.get("texto_original", "")[:55]
    tag = f"[{b['tipo']}]" + (" [SEM FALA]" if b["status"] == "CENA_SEM_FALA" else "")
    print(f"  {tag} [{b['start']}s-{b['end']}s]{lp} {txt}")

if 'salvar_no_drive' in globals():
    print('💾 Salvando progresso no Drive...')
    salvar_no_drive(TRANSCRICAO_PATH, 'KAGGLE/AUDIO_DUB/transcricao_raw.json')

cell_end(2, "done", "Setup Inicial + Drive + Whisper concluido")

In [ ]:
cell_start(3, "Celula 3") 
# @title 🕵️ 3. Identificação do Anime e Personagens
# @markdown Analisa o texto transcrito (zh/ja) para identificar o anime,
# @markdown protagonista e personagens antes da tradução.
# @markdown Modelos e fallback configurados na Célula 1.

import json
import os

IDENTIFICACAO_PATH = f"{BASE_PATH}/identificacao_anime.json"

if os.path.exists(IDENTIFICACAO_PATH):
    print("📁 Identificação já existe. Carregando cache...")
    with open(IDENTIFICACAO_PATH, 'r', encoding='utf-8') as f:
        anime_info = json.load(f)
    selected_anime = anime_info.get("title", "Desconhecido")
    protagonist    = anime_info.get("protagonist", "Narrador")
    print("✅ Cache carregado.")
else:
    if 'final_blocks' not in globals() or not final_blocks:
        try:
            with open(TRANSCRICAO_PATH, "r", encoding='utf-8') as f:
                final_blocks = json.load(f)
            print("⚠️ Transcrição carregada do arquivo.")
        except:
            raise Exception("❌ Rode a Célula 2 primeiro.")

    textos_com_fala = [b['texto_original'] for b in final_blocks
                       if b.get("texto_original", "").strip()]
    full_text_sample = " ".join(textos_com_fala[:60])

    if not full_text_sample.strip():
        raise Exception("❌ Nenhum texto original encontrado nos blocos.")

    print("🕵️ Analisando texto para identificar o anime...")

    identify_prompt = f"""
Analyze the following anime/movie transcript deeply.
The text may be in Chinese (Mandarin narration) and/or Japanese (character dialogue).

TEXT SAMPLE:
"{full_text_sample[:5000]}"

TASK:
1. Identify the **Anime/Movie Title** (use the most common English/Romaji name).
2. Identify the **main protagonist** — the character performing most actions in THIS specific text.
   - CRITICAL: Do not just pick the main franchise character.
   - Example: If the anime is "Naruto" but the text focuses on "Sasuke seeking revenge", pick Sasuke.
3. List ALL character names that appear or are referenced.
4. Brief synopsis of what happens in this specific segment.

OUTPUT FORMAT (JSON ONLY):
{{
    "title": "Anime Name",
    "title_jp": "Japanese Title (if known)",
    "protagonist": "Character Name",
    "characters": ["char1", "char2", "char3"],
    "synopsis": "Brief description of what happens",
    "reason": "Why you identified this anime and protagonist"
}}
"""

    anime_info = None
    try:
        # Tenta Gemini com fallback automático por quota (cadeias definidas na Cel1)
        response = gemini_generate(
            model_chain=MODELS_IDENTIFICACAO,
            contents=identify_prompt,
            config={'response_mime_type': 'application/json', 'temperature': 0.1},
            task_name="Identificacao"
        )
        anime_info = json.loads(response.text)

    except RuntimeError as e:
        # Todos Gemini falharam — tenta OpenAI se disponível
        if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
            print("  Tentando OpenAI como fallback final...")
            try:
                resp = client_openai.chat.completions.create(
                    model=OPENAI_FALLBACK_MODEL,
                    messages=[{"role": "user", "content": identify_prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.1
                )
                anime_info = json.loads(resp.choices[0].message.content)
                print("  OpenAI fallback OK.")
            except Exception as oe:
                print(f"  OpenAI também falhou: {oe}")

    except Exception as e:
        print(f"❌ Erro fatal na identificação: {e}")

    if not anime_info:
        anime_info = {
            "title": "Desconhecido",
            "protagonist": "Narrador",
            "characters": [],
            "synopsis": "",
            "reason": "Falha em todos os modelos disponíveis."
        }

    selected_anime = anime_info.get("title", "Desconhecido")
    protagonist    = anime_info.get("protagonist", "Narrador")

    with open(IDENTIFICACAO_PATH, 'w', encoding='utf-8') as f:
        json.dump(anime_info, f, ensure_ascii=False, indent=4)
    print(f"💾 Salvo: {IDENTIFICACAO_PATH}")

print(f"""
╔════════════════════════════════════════════════════════╗
║            🎬 ANIME IDENTIFICADO                       ║
╠════════════════════════════════════════════════════════╣
║  Título       : {selected_anime}
║  Protagonista : {protagonist}
║  Personagens  : {', '.join(anime_info.get('characters', [])[:8])}
║  Sinopse      : {anime_info.get('synopsis', 'N/A')[:80]}...
╚════════════════════════════════════════════════════════╝
""")

if 'salvar_no_drive' in globals():
    salvar_no_drive(IDENTIFICACAO_PATH, 'KAGGLE/AUDIO_DUB/identificacao_anime.json')

cell_end(3, "done", "Celula 3 concluido")

In [ ]:
cell_start(4, "Tradução & Storytelling")
# @title 🌍 4. Tradução Inteligente (com contexto do anime identificado)
# @markdown Traduz TODOS os blocos para português usando o nome do anime
# @markdown e personagens como contexto para evitar erros de nomes/termos.
# @markdown
# @markdown  NARRACAO (zh): Forçado em 3ª pessoa, mesmo que o original seja 1ª.
# @markdown  CENA (ja): Preserva a pessoa gramatical do original (1ª ou 3ª).
# @markdown Modelos e fallback configurados na Célula 1.

import json
import time
import os

TARGET_LANGUAGE  = "Portuguese (Brazil)"

TRADUCAO_PATH    = f"{BASE_PATH}/traducao_simplificada.json"
TRANSCRICAO_PATH = f"{BASE_PATH}/transcricao_raw.json"

_anime_title = globals().get('selected_anime', 'Desconhecido')
_protagonist  = globals().get('protagonist', 'Narrador')
_characters   = anime_info.get('characters', []) if 'anime_info' in globals() else []
_char_list    = ', '.join(_characters) if _characters else 'Desconhecidos'

print(f"Contexto: {_anime_title} | Protagonista: {_protagonist}")
print(f"Personagens: {_char_list}")

if should_run('translate'):
    if os.path.exists(TRADUCAO_PATH):
        print("Traducao ja existe. Carregando cache...")
        with open(TRADUCAO_PATH, 'r', encoding='utf-8') as f:
            final_blocks = json.load(f)
        print(f"{len(final_blocks)} blocos carregados.")
    else:
        if 'final_blocks' not in globals() or not final_blocks:
            try:
                with open(TRANSCRICAO_PATH, "r", encoding='utf-8') as f:
                    final_blocks = json.load(f)
                print("Backup carregado do arquivo.")
            except:
                raise Exception("Rode a Celula 2 primeiro.")

        blocos_para_traduzir = [
            b for b in final_blocks
            if b.get("texto_original", "").strip() and not b.get("translated_text", "").strip()
        ]
        blocos_cena_sem_fala = [
            b for b in final_blocks
            if b["tipo"] == "CENA" and not b.get("texto_original", "").strip()
        ]

        for b in blocos_cena_sem_fala:
            b["translated_text"] = ""
            b["status"] = "CENA_SEM_FALA"

        narracao_list  = [b for b in blocos_para_traduzir if b["tipo"] == "NARRACAO"]
        cena_fala_list = [b for b in blocos_para_traduzir if b["tipo"] == "CENA"]

        total_lines = len(blocos_para_traduzir)
        print(f"Traduzindo {total_lines} blocos | {len(narracao_list)} narracoes | {len(cena_fala_list)} falas de cena")

        # ──────────────────────────────────────────────────────────────────────────
        # PROMPTS SEPARADOS POR TIPO
        # ──────────────────────────────────────────────────────────────────────────

        SYSTEM_NARRACAO = f"""
    You are a professional dubbing scriptwriter specializing in anime narration for viral short-form videos.
    Your task is to translate Chinese Mandarin narration blocks into {TARGET_LANGUAGE}.

    CONTEXT: Anime "{_anime_title}" | Main character: {_protagonist} | Known cast: {_char_list}.

    ─── TRANSLATION RULES ───────────────────────────────────────────────────────

    1. PERSON — Always Third Person. Never First Person.
       - "我" / "我们" → "ele/ela/eles", never "eu/nós".
       - Example: "我看到了敌人" → "Ele avistou o inimigo" ✅ | "Eu vi o inimigo" ❌

    2. LANGUAGE LEVEL — Simple, clear, everyday vocabulary (A2/B1).
       - Write for the EAR, not the page. Short sentences, natural spoken rhythm.
       - Avoid academic, archaic, or overly formal words.

    3. REPETITION — Never repeat the same word or phrase twice in a row across consecutive lines.
       - Vary pronouns and synonyms naturally.

    4. PROPER NOUNS & LOCALIZATION — Keep the original Japanese name/term only the FIRST time it is introduced in the narration.
       - For subsequent occurrences, do not repeat the complex Japanese names or terms. Instead, substitute them naturally with pronouns, relationship words, or roles to make the storytelling easier to follow (e.g., "seu irmão", "o protagonista", "o jovem", "o garoto", "o guerreiro", "seu amigo").
       - Kinship/role words are NOT proper nouns: "pai", "mãe", "mestre", "rei", "imperador", etc.
         Translate them normally — do NOT keep "Father", "Master", "Lord" in English mid-sentence.
       - Example: "Father said..." → "Seu pai disse..." ✅ | "Father disse..." ❌

    5. NARRATIVE FLOW — Cinema narration, not a fact report.
       - Never start two consecutive blocks with the same pronoun/subject.
       - Use connectors: "Enquanto isso", "Diante disso", "Sem hesitar", "Para sua surpresa", "Anos antes", "Naquele momento" etc.
       - Merge related ideas in a single block into one flowing sentence.
       - Vary structure: participial phrases, inverted subjects, temporal clauses.

    6. SIMPLE AND DIRECT TONE:
       - Use simple words that everyone understands. Avoid fancy or uncommon words.
       - Prefer common verbs: "foi", "fez", "disse", "viu" over "avançou", "sucumbiu", "impôs".
       - BAD:  "Ao adentrar o recinto, a melancolia apoderou-se de sua psique."
       - GOOD: "Quando ele entrou na sala, ficou muito triste."

    7. SYNC — Keep the translated length close to the original for audio timing.
       One block = one line. Do not merge or split blocks.

    ─── OUTPUT ──────────────────────────────────────────────────────────────────

    Respond with JSON only. No explanation, no markdown, no extra keys.
    {{"translations": ["string 1", "string 2", ...]}}
    The array MUST contain exactly {{total}} items in the original order.
    """

        SYSTEM_CENA = f"""
    You are a professional dubbing scriptwriter for viral anime videos.
    Translate the input list from Japanese into {TARGET_LANGUAGE}.

    CONTEXT: Anime "{_anime_title}" | Main character: {_protagonist} | Known cast: {_char_list}.

    TRANSLATION RULES FOR DIALOGUE BLOCKS (Character Dialogue):
    1. PERSON: Preserve the grammatical person of the original speaker.
       - If the character speaks in First Person (僕/俺/私 = I/me), keep it as First Person in PT-BR.
       - If the character narrates in Third Person, keep Third Person.
       - DO NOT forcefully change "Eu" to "Ele" — these are authentic character voices.
    2. AUTHENTICITY: Keep the character's voice, tone and personality.
       - Aggressive character → speak with aggression.
       - Timid character → speak softly/hesitantly.
    3. VOCABULARY: Colloquial Brazilian Portuguese. Natural speech, not literary.
    4. SYNC: Keep length similar to the original for lip-sync timing.
    5. NAMES & LOCALIZATION: Keep the original Japanese name/term only the FIRST time it is introduced.
       - For subsequent occurrences, avoid repeating complex Japanese names or terms. Instead, substitute them naturally with pronouns, relationship words, or roles to make it easy for the audience to follow (e.g., "seu irmão", "o garoto", "sua amiga", "ele").

    OUTPUT FORMAT (JSON ONLY):
    {{ "translations": ["string 1", "string 2", ...] }}
    The list MUST have exactly {{total}} items in the same order.
    """

        # ──────────────────────────────────────────────────────────────────────────
        # FUNÇÃO DE TRADUÇÃO COM FALLBACK
        # ──────────────────────────────────────────────────────────────────────────
        def _traduzir_chunk(chunk, system_instruction, task_label):
            """Traduz um chunk de blocos com fallback OpenAI para erros do Gemini 3.1 Pro."""
            source_texts = [b["texto_original"] for b in chunk]
            total = len(source_texts)
            user_payload = json.dumps({"sentences": source_texts}, ensure_ascii=False)
            full_prompt  = f"{system_instruction}\n\nINPUT DATA:\n{user_payload}"
            translated_list = None
            gemini_failed = False

            try:
                response = gemini_generate(
                    model_chain=MODELS_TRADUCAO,
                    contents=full_prompt,
                    config={'response_mime_type': 'application/json', 'temperature': 0.2},
                    task_name=task_label
                )
                data = json.loads(response.text)
                if isinstance(data, list): translated_list = data
                elif "translations" in data: translated_list = data["translations"]
                else: translated_list = list(data.values())[0]
            except Exception as e:
                print(f"  [ERRO] Gemini falhou ({task_label}): {e}")
                gemini_failed = True

            # Fallback OpenAI quando Gemini falhou (qualquer erro incluindo JSON invalido)
            if (gemini_failed or translated_list is None) and USE_OPENAI_FALLBACK and client_openai:
                print(f"  [FALLBACK] Usando OpenAI {OPENAI_FALLBACK_MODEL} para {task_label}...")
                try:
                    resp = client_openai.chat.completions.create(
                        model=OPENAI_FALLBACK_MODEL,
                        messages=[
                            {"role": "system", "content": system_instruction},
                            {"role": "user",   "content": user_payload}
                        ],
                        response_format={"type": "json_object"}, temperature=0.2
                    )
                    data = json.loads(resp.choices[0].message.content)
                    if isinstance(data, list): translated_list = data
                    elif "translations" in data: translated_list = data["translations"]
                    else: translated_list = list(data.values())[0]
                    print(f"  [OK] OpenAI fallback OK ({task_label}).")
                except Exception as oe:
                    print(f"  [ERRO] OpenAI tambem falhou: {oe}")

            return translated_list, total

        def traduzir_batch(blocks, system_instruction_template, task_label):
            if not blocks: return
            # Determina tamanho do chunk: se anime identificado usa BATCH_SIZE, senão 60
            _anime_ok = globals().get('selected_anime', 'Desconhecido') != 'Desconhecido'
            _chunk_sz  = globals().get('BATCH_SIZE', 50) if _anime_ok else 60
            total = len(blocks)
            system_instruction = system_instruction_template.replace("{total}", str(_chunk_sz))
            print(f"  Chunks de {_chunk_sz} | Total: {total} blocos")

            for i in range(0, total, _chunk_sz):
                chunk = blocks[i: i + _chunk_sz]
                chunk_label = f"{task_label}-chunk{i//  _chunk_sz + 1}"
                # Ajusta {total} para o tamanho real do chunk
                sys_inst = system_instruction_template.replace("{total}", str(len(chunk)))
                translated_list, expected = _traduzir_chunk(chunk, sys_inst, chunk_label)

                if not translated_list:
                    for block in chunk:
                        block["translated_text"] = f"[FALHA] {block['texto_original']}"
                        block["status"] = "FAILED"
                    continue

                if len(translated_list) != expected:
                    print(f"  Recebidos {len(translated_list)} vs esperados {expected}. Ajustando...")

                for j, block in enumerate(chunk):
                    block["translated_text"] = (
                        translated_list[j] if j < len(translated_list)
                        else f"[FALHA] {block['texto_original']}"
                    )
                    block["status"] = "TRANSLATED"


                print(f"  Chunk {i//  _chunk_sz + 1}: {len(chunk)} blocos traduzidos.")

        # ──────────────────────────────────────────────────────────────────────────
        # EXECUÇÃO
        # ──────────────────────────────────────────────────────────────────────────
        start = time.time()

        if narracao_list:
            print(f"\nTraduzindo {len(narracao_list)} narracoes (zh -> pt, 3a pessoa forcada)...")
            traduzir_batch(narracao_list, SYSTEM_NARRACAO, "Narracao-zh")

        if cena_fala_list:
            print(f"\nTraduzindo {len(cena_fala_list)} falas de cena (ja -> pt, pessoa original)...")
            traduzir_batch(cena_fala_list, SYSTEM_CENA, "Cena-ja")

        elapsed = time.time() - start
        print(f"\nTraducao concluida em {elapsed:.2f}s.")

        with open(TRADUCAO_PATH, 'w', encoding='utf-8') as f:
            json.dump(final_blocks, f, ensure_ascii=False, indent=4)
        print(f"Salvo: {TRADUCAO_PATH}")

        print("\nPreview (3 narracoes):")
        print("-" * 50)
        for b in [x for x in final_blocks if x["tipo"] == "NARRACAO"][:3]:
            print(f"Original : {b['texto_original']}")
            print(f"Traduzido: {b['translated_text']}")
            print("-" * 50)

        print("\nPreview (3 falas de cena):")
        print("-" * 50)
        for b in [x for x in final_blocks if x["tipo"] == "CENA" and x.get("translated_text")][:3]:
            print(f"Original : {b['texto_original']}")
            print(f"Traduzido: {b['translated_text']}")
            print("-" * 50)

    # Geração automática da introdução de 10s e seleção de 2 cenas
    print("\n[INTRO] Gerando roteiro da introdução e selecionando cenas...")
    INTRO_PROMPT = f"""
You are a scriptwriter and video editor. Based on the following translated anime recap script, please generate:
1. A short, highly engaging video introduction (hook) to grab the viewer's attention.
   - It MUST start with a hook based on the script's peak moments, followed by a friendly/casual greeting.
   - Greeting style: "Fala família..." (Brazilian Portuguese casual style).
   - Target duration: Exactly 10 seconds of spoken voice. So the text MUST be very short, around 100-130 characters maximum!
   - Example style: "Reencarnado como um cachorro, ele terá que fugir do canil que o espreita... Fala família, se prepara que nesse vídeo o que não falta é cachorrada!"
2. Choose exactly 2 high-impact/interesting scene timestamps from the script to compose the introduction.
   - For each scene, find a good timestamp block in the script (look at the 'start' and 'end' time of the blocks).
   - The scene timestamps will be used to cut 5-second video clips to make a 10-second intro video.
   - Choose timestamps from the script blocks where there is action, drama, or something visual happening.

Input script:
{json.dumps([{ 'id': b['id'], 'tipo': b['tipo'], 'text': b.get('translated_text') or b.get('texto_original', ''), 'start': b['start'], 'end': b['end'] } for b in final_blocks], ensure_ascii=False)}

Respond with JSON only, using this exact format:
{{
  "intro_text": "text for the 10 seconds introduction speech",
  "scenes": [
    {{ "start": float, "end": float, "description": "brief description of scene 1" }},
    {{ "start": float, "end": float, "description": "brief description of scene 2" }}
  ]
}}
"""
    try:
        from google.genai import types
        cfg = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema={
                "type": "OBJECT",
                "properties": {
                    "intro_text": {"type": "STRING"},
                    "scenes": {
                        "type": "ARRAY",
                        "items": {
                            "type": "OBJECT",
                            "properties": {
                                "start": {"type": "NUMBER"},
                                "end": {"type": "NUMBER"},
                                "description": {"type": "STRING"}
                            },
                            "required": ["start", "end"]
                        }
                    }
                },
                "required": ["intro_text", "scenes"]
            }
        )
        resp = gemini_generate(MODELS_TRADUCAO, INTRO_PROMPT, config=cfg, task_name="Geração de Intro")
        intro_data = json.loads(resp.text)
        print("[INTRO] Intro gerada com sucesso:", intro_data)
    except Exception as ie:
        print("[INTRO ERROR] Falha ao gerar intro pelo Gemini, usando fallback local:", ie)
        intro_data = {
            "intro_text": "Se prepara, família, que hoje o vídeo está sensacional! Vamos acompanhar essa história insana do início ao fim.",
            "scenes": [
                {"start": 10.0, "end": 15.0},
                {"start": 40.0, "end": 45.0}
            ]
        }
    
    INTRO_INFO_PATH = f"{BASE_PATH}/intro_info.json"
    with open(INTRO_INFO_PATH, 'w', encoding='utf-8') as f:
        json.dump(intro_data, f, ensure_ascii=False, indent=4)
    print(f"Salvo: {INTRO_INFO_PATH}")
    
    if 'salvar_no_drive' in globals():
        salvar_no_drive(TRADUCAO_PATH, 'KAGGLE/AUDIO_DUB/traducao_simplificada.json')
        salvar_no_drive(INTRO_INFO_PATH, 'KAGGLE/AUDIO_DUB/intro_info.json')
    cell_end(4, 'done', 'Tradução e intro concluídas')
else:
    print('Passo de traducao ignorado para esta task.')
    cell_end(4, 'done', 'Tradução ignorada')


In [ ]:
cell_start(5, "Dividir Roteiro e Reportar Main Done")
# Divide traducao_simplificada.json em 4 partes e faz upload ao Drive.
# Reporta step_omni = "main_done" via webhook para o controller iniciar os 4 TTS.
import json
import os
import math

TRADUCAO_PATH = f"{BASE_PATH}/traducao_simplificada.json"

if not os.path.exists(TRADUCAO_PATH):
    raise FileNotFoundError(f"traducao_simplificada.json nao encontrado em {TRADUCAO_PATH}")

with open(TRADUCAO_PATH, 'r', encoding='utf-8') as f:
    all_blocks = json.load(f)

total = len(all_blocks)
num_parts = 4
chunk_size = math.ceil(total / num_parts)

print(f"Dividindo {total} blocos em {num_parts} partes (chunk_size={chunk_size})...")

for i in range(1, num_parts + 1):
    start = (i - 1) * chunk_size
    end = min(i * chunk_size, total)
    part_blocks = all_blocks[start:end]
    part_path = f"{BASE_PATH}/roteiro_pt{i}.json"
    with open(part_path, 'w', encoding='utf-8') as f:
        json.dump(part_blocks, f, ensure_ascii=False, indent=4)
    print(f"  Parte {i}: {len(part_blocks)} blocos (blocos {start}-{end-1}) -> {part_path}")
    if 'salvar_no_drive' in globals():
        salvar_no_drive(part_path, f"KAGGLE/AUDIO_DUB/roteiro_pt{i}.json")
        print(f"  Upload Drive: KAGGLE/AUDIO_DUB/roteiro_pt{i}.json")

print("\nDivisao concluida! Reportando main_done...")
cell_end(5, "done", "Roteiro dividido em 4 partes")
report_step("main_done", "Omni-Main concluido. Aguardando TTS paralelos.")
